<a href="https://colab.research.google.com/github/MisterCioffi/SteganoGAN/blob/master/Provarumorestega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Clona il repository
!git clone https://github.com/MisterCioffi/SteganoGAN.git

# 2. Entra nella cartella del progetto
%cd SteganoGAN

# 3. Rimuovi i limiti superiori stringenti dal file setup.py
with open('setup.py', 'r') as f:
    setup_content = f.read()

# Rimuoviamo i vincoli superiori che rompono Colab
setup_content = setup_content.replace('<2.5.0', '')
setup_content = setup_content.replace('<1.2.0', '')
setup_content = setup_content.replace('<1.16.0', '')
setup_content = setup_content.replace('<8.0.0', '')
setup_content = setup_content.replace('<2.0.0', '')

with open('setup.py', 'w') as f:
    f.write(setup_content)

print("setup.py patchato con successo!")

Cloning into 'SteganoGAN'...
remote: Enumerating objects: 1302, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 1302 (delta 16), reused 37 (delta 11), pack-reused 1258 (from 1)
Receiving objects: 100% (1302/1302), 44.15 MiB | 23.24 MiB/s, done.
Resolving deltas: 100% (538/538), done.
/content/SteganoGAN
setup.py patchato con successo!


In [4]:
# Installa le dipendenze e il pacchetto
!pip install -e .

Obtaining file:///content/SteganoGAN
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for reedsolo: filename=reedsolo-0.3-py3-none-any.whl size=5301 sha256=5ebc4b938f2a07873cadd5ff47e2fa6d341179b20f1d6a322c53cfc8ed8c3753
  Stored in directory: /root/.cache/pip/wheels/5c/14/ad/f69c34ef121e2c2ef5ae1123d4efaec9f7e246ad734186ffa3
Successfully built reedsolo
  Running setup.py develop for steganogan


In [6]:
from PIL import Image

def ritaglia_al_centro(percorso_input, percorso_output, dimensione=512):
    """
    Prende un'immagine, la ritaglia esattamente al centro con le dimensioni richieste
    e la salva come nuovo file.
    """
    # 1. Apri l'immagine
    img = Image.open(percorso_input)
    larghezza, altezza = img.size

    print(f"Dimensioni originali: {larghezza}x{altezza}")

    # 2. Verifica che l'immagine sia grande abbastanza
    if larghezza < dimensione or altezza < dimensione:
        print("Errore: L'immagine è più piccola di 512x512, usane una più grande!")
        return

    # 3. Calcola le coordinate matematiche per il ritaglio centrale
    sinistra = (larghezza - dimensione) / 2
    alto = (altezza - dimensione) / 2
    destra = (larghezza + dimensione) / 2
    basso = (altezza + dimensione) / 2

    # 4. Esegui il taglio
    img_ritagliata = img.crop((sinistra, alto, destra, basso))

    # 5. Salva la nuova immagine
    img_ritagliata.save(percorso_output)
    print(f"Fatto! Immagine perfetta salvata in '{percorso_output}' con dimensioni: {img_ritagliata.size}")

In [7]:
# --- ESEGUI IL TAGLIO ---
# Inserisci il nome della tua foto reale originale e il nome del nuovo file
ritaglia_al_centro('research/input.png', 'research/input_512.png', 512)

Dimensioni originali: 2040x1152
Fatto! Immagine perfetta salvata in 'research/input_512.png' con dimensioni: (512, 512)


In [8]:
import os
import torch
import torch.optim
import torch.optim.adam
import steganogan.models
from steganogan import SteganoGAN
import numpy as np
import skimage.io as io

# --- 1. PATCH PER L'OTTIMIZZATORE ADAM (Livello 1 e 2) ---
_original_optimizer_setstate = torch.optim.Optimizer.__setstate__
def _safe_setstate(self, state):
    if isinstance(state, dict) and 'defaults' not in state:
        state['defaults'] = {}
    try:
        _original_optimizer_setstate(self, state)
    except Exception:
        pass
torch.optim.Optimizer.__setstate__ = _safe_setstate

# NOVITÀ: Patch specifica per Adam per l'errore param_groups
_original_adam_setstate = torch.optim.Adam.__setstate__
def _safe_adam_setstate(self, state):
    if not hasattr(self, 'param_groups'):
        self.param_groups = []  # Gli diamo una lista vuota per non farlo arrabbiare
    try:
        _original_adam_setstate(self, state)
    except Exception:
        pass
torch.optim.Adam.__setstate__ = _safe_adam_setstate


# --- 2. PATCH PER LA SICUREZZA DI PYTORCH 2.6+ ---
@classmethod
def custom_load(cls, architecture=None, path=None, cuda=True, verbose=False):
    if architecture and not path:
        model_name = '{}.steg'.format(architecture)
        pretrained_path = os.path.join(os.path.dirname(steganogan.models.__file__), 'pretrained')
        path = os.path.join(pretrained_path, model_name)
    elif (architecture is None and path is None) or (architecture and path):
        raise ValueError('Please provide either an architecture or a path to pretrained model.')

    # weights_only=False per bypassare il blocco di sicurezza
    steganogan_model = torch.load(path, map_location='cpu', weights_only=False)
    steganogan_model.verbose = verbose

    steganogan_model.encoder.upgrade_legacy()
    steganogan_model.decoder.upgrade_legacy()
    steganogan_model.critic.upgrade_legacy()

    steganogan_model.set_device(cuda)
    return steganogan_model

SteganoGAN.load = custom_load


# --- 3. IL TEST FINALE ---
print("Caricamento del modello (con Super Patch 2.0 attiva)...")
steganogan_model = SteganoGAN.load(architecture='dense')

# Ora proviamo finalmente il Logo!
cover_image_path = 'research/input_512.png'
stego_image_path = 'output.png'
messaggio_segreto = "https://it.wikipedia.org/wiki/Pongo_pygmaeus"

print("\nNascondo il messaggio nel Logo...")
steganogan_model.encode(cover_image_path, stego_image_path, messaggio_segreto)
print(f"Immagine salvata in: {stego_image_path}")

x = np.float64(io.imread(cover_image_path))
z = np.float64(io.imread(stego_image_path))
MSE = np.mean((x-z)**2)

print("MSE: ", MSE)

print("\nEstraggo il messaggio dal Logo modificato...")
messaggio_decodificato = steganogan_model.decode(stego_image_path)

print("\n--- RISULTATO ---")
print(f"Messaggio estratto: {messaggio_decodificato}")

Caricamento del modello (con Super Patch 2.0 attiva)...


/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.conv.Conv2d' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.activation.LeakyReLU' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.


Nascondo il messaggio nel Logo...
Immagine salvata in: output.png
MSE:  12.97656504313151

Estraggo il messaggio dal Logo modificato...

--- RISULTATO ---
Messaggio estratto: https://it.wikipedia.org/wiki/Pongo_pygmaeus


In [10]:
import torchvision.transforms as transforms
from PIL import Image

def interroga_il_critic(percorso_immagine, modello):
    """
    Legge un'immagine e chiede al Critic di valutarla.
    Restituisce un numero: > 0 (Tende al Reale), < 0 (Tende al Falso/Modificato)
    """
    # 1. Carica l'immagine a colori
    img = Image.open(percorso_immagine).convert('RGB')

    # 2. Trasforma l'immagine in Tensore e normalizza i pixel tra 0 e 1
    trasformazione = transforms.ToTensor()
    img_tensor = trasformazione(img).unsqueeze(0) # unsqueeze aggiunge la dimensione 'Batch' (1, 3, H, W)

    # 3. Sposta i dati sulla CPU o GPU (in base a dove sta girando il modello)
    img_tensor = img_tensor.to(modello.device)

    # 4. Chiedi il parere al Critic senza allenarlo (no_grad)
    with torch.no_grad():
        score_tensor = modello.critic(img_tensor)

    # 5. Fai la media per ottenere il voto finale (torch.mean)
    voto_finale = score_tensor.mean().item()
    return voto_finale

In [14]:
import torch
import torchvision.transforms as transforms

def estrai_bit_crudi(modello, percorso_img, quanti_bit=400):
    """
    Bypassa il decode() ufficiale e forza l'estrazione dei bit grezzi
    direttamente dalla rete neurale.
    """
    # 1. Carica l'immagine in memoria
    img = Image.open(percorso_img).convert('RGB')

    # 2. Trasforma l'immagine in Tensore per la rete neurale
    img_tensor = transforms.ToTensor()(img).unsqueeze(0).to(modello.device)

    # 3. Interroga brutalmente il Decoder
    with torch.no_grad():
        # Il decoder restituisce numeri positivi (tende a 1) e negativi (tende a 0)
        mappa_probabilita = modello.decoder(img_tensor)

    # 4. Appiattisci la mappa in una fila singola di bit (0 o 1)
    bit_predetti = (mappa_probabilita > 0).cpu().numpy().flatten().astype(int)

    # 5. Restituisci solo i primi N bit
    bit_estratti = bit_predetti[:quanti_bit]

    # Convertiamo l'array in una stringa visibile (es. "0110010...")
    stringa_bit = "".join(map(str, bit_estratti))

    print(f"Estrazione Raw per {percorso_img}:")
    print(f"{stringa_bit[:100]}... (Mostro i primi 100 bit)")

    return stringa_bit

# Testiamolo sull'immagine originale e su quella con il rumore gaussiano minimo
print("--- TEST ESTRAZIONE CRUDA (RAW) ---")
bit_originali = estrai_bit_crudi(steganogan_model, 'output.png')
bit_corrotti = estrai_bit_crudi(steganogan_model, 'test_rumore_sigma_0.5.png')

# Contiamo quanti bit sono effettivamente cambiati
differenze = sum(1 for a, b in zip(bit_originali, bit_corrotti) if a != b)
accuratezza = 1.0 - (differenze / len(bit_originali))

print(f"\nSu 400 bit estratti, il rumore gaussiano ne ha capovolti {differenze}.")
print(f"L'accuratezza reale della rete a Sigma 0.5 è del {accuratezza*100:.1f}%")

--- TEST ESTRAZIONE CRUDA (RAW) ---
Estrazione Raw per output.png:
1011010011101011000001001000111101111100010000001111011000111101010100011011011010101011111110110010... (Mostro i primi 100 bit)
Estrazione Raw per test_rumore_sigma_0.5.png:
1011010011101011000001001000111111111100010000001111011000111101010100001011011010101010111110110010... (Mostro i primi 100 bit)

Su 400 bit estratti, il rumore gaussiano ne ha capovolti 24.
L'accuratezza reale della rete a Sigma 0.5 è del 94.0%


In [16]:
import torch
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

def analizza_resistenza_raw_multiplo(modello, percorso_png, quanti_bit=1000):
    """
    Applica vari livelli di rumore e misura la vera accuratezza a livello di bit
    bypassando la decodifica del testo.
    """
    # 1. Carica l'immagine pulita e crea una base in NumPy
    img_pulita = Image.open(percorso_png).convert('RGB')
    img_arr = np.array(img_pulita, dtype=np.float32)

    # 2. Estrai la "Verità" (i bit originali prima del rumore)
    img_tensor_pulita = transforms.ToTensor()(img_pulita).unsqueeze(0).to(modello.device)
    with torch.no_grad():
        bit_puliti = (modello.decoder(img_tensor_pulita) > 0).cpu().numpy().flatten()[:quanti_bit]

    # I livelli di rumore da testare
    livelli_rumore = [0.5, 1.0, 2.0, 5.0, 10.0]

    print(f"--- TEST DI ROBUSTEZZA RAW (Campione: {quanti_bit} bit) ---\n")

    for sigma in livelli_rumore:
        # 3. Applica il rumore all'immagine
        rumore = np.random.normal(loc=0.0, scale=sigma, size=img_arr.shape)
        img_corrotta_arr = np.clip(img_arr + rumore, 0, 255).astype(np.uint8)

        # 4. Converti in Tensore per la rete neurale
        img_corrotta_pil = Image.fromarray(img_corrotta_arr)
        img_tensor_corrotta = transforms.ToTensor()(img_corrotta_pil).unsqueeze(0).to(modello.device)

        # 5. Estrai i bit crudi dall'immagine rumorosa
        with torch.no_grad():
            bit_corrotti = (modello.decoder(img_tensor_corrotta) > 0).cpu().numpy().flatten()[:quanti_bit]

        # 6. Confronta gli array per vedere quanti bit sono diversi
        differenze = np.sum(bit_puliti != bit_corrotti)
        accuratezza = 100.0 * (1.0 - (differenze / quanti_bit))

        print(f"Sigma {sigma:4.1f} | Bit capovolti: {differenze:4d} / {quanti_bit} | Accuratezza Reale: {accuratezza:.1f}%")

# --- ESECUZIONE ---
analizza_resistenza_raw_multiplo(steganogan_model, 'output.png', 1000)

--- TEST DI ROBUSTEZZA RAW (Campione: 1000 bit) ---

Sigma  0.5 | Bit capovolti:   99 / 1000 | Accuratezza Reale: 90.1%
Sigma  1.0 | Bit capovolti:  137 / 1000 | Accuratezza Reale: 86.3%
Sigma  2.0 | Bit capovolti:  200 / 1000 | Accuratezza Reale: 80.0%
Sigma  5.0 | Bit capovolti:  316 / 1000 | Accuratezza Reale: 68.4%
Sigma 10.0 | Bit capovolti:  387 / 1000 | Accuratezza Reale: 61.3%


In [18]:
import numpy as np
from PIL import Image

def testa_rumore_con_header_protetto(modello, percorso_png, messaggio_originale):
    """
    Applica rumore gaussiano all'immagine, ma 'protegge' le prime righe in alto
    dove si trova l'header della libreria, evitando il crash.
    """
    # 1. Carica l'immagine
    img_arr = np.array(Image.open(percorso_png).convert('RGB'), dtype=np.float32)

    # Scegliamo un livello di rumore (Sigma 2.0 prima faceva crashare tutto istantaneamente)
    sigma = 2.0

    print(f"--- TEST RUMORE (SIGMA {sigma}) CON HEADER PROTETTO ---")

    # 2. Generiamo il rumore per tutta l'immagine
    rumore = np.random.normal(loc=0.0, scale=sigma, size=img_arr.shape)

    # 3. LA TUA IDEA: Proteggiamo le prime 15 righe dell'immagine azzerando il rumore
    # img_arr.shape è (Altezza, Larghezza, Canali).
    # rumore[:15, :, :] prende le prime 15 righe (Y da 0 a 14) e tutti i pixel X e Colori.
    rumore[:15, :, :] = 0.0

    # 4. Applichiamo il rumore "mascherato"
    img_corrotta = np.clip(img_arr + rumore, 0, 255).astype(np.uint8)

    # 5. Salviamo l'immagine
    nome_file = "test_header_protetto.png"
    Image.fromarray(img_corrotta).save(nome_file)

    # 6. Proviamo il decode ufficiale!
    try:
        testo_estratto = modello.decode(nome_file)
        print("\n🟢 SUCCESSO! Il decoder ha letto l'header e NON è crashato.")
        print("\nTesto Originale :", messaggio_originale)
        print("Testo Estratto  :", testo_estratto)

    except Exception as e:
        print(f"\n🔴 CRASH: {e} (Evidentemente l'header va oltre le prime 15 righe!)")

# --- ESECUZIONE ---
messaggio_segreto = "https://it.wikipedia.org/wiki/Pongo_pygmaeus"
testa_rumore_con_header_protetto(steganogan_model, 'output.png', messaggio_segreto)

--- TEST RUMORE (SIGMA 2.0) CON HEADER PROTETTO ---

🔴 CRASH: Failed to find message. (Evidentemente l'header va oltre le prime 15 righe!)


In [19]:
import torch
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

def stringa_a_bit(testo):
    """Converte una stringa in una lista di bit (0 e 1)"""
    return [int(b) for b in ''.join(format(ord(c), '08b') for c in testo)]

def bit_a_stringa(bits):
    """Converte una lista di bit in testo, sostituendo i danni gravi con '?'"""
    caratteri = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) == 8:
            valore = int(''.join(map(str, byte)), 2)
            # Stampiamo solo ASCII standard per evitare crash della console
            if 32 <= valore <= 126:
                caratteri.append(chr(valore))
            else:
                caratteri.append('?')
    return ''.join(caratteri)

def hacking_steganografico(modello, percorso_input, testo_segreto, sigma_rumore=5.0):
    print("--- INIZIO HACKING STEGANOGRAFICO (Forza Bruta + Voto) ---")
    print(f"Livello di Rumore (Sigma Gaussiano): {sigma_rumore}\n")

    # 1. Carichiamo l'immagine come Tensore PyTorch
    img_pil = Image.open(percorso_input).convert('RGB')
    img_tensor = transforms.ToTensor()(img_pil).unsqueeze(0).to(modello.device)

    # 2. Scopriamo la vera capacità (interroghiamo il Decoder per farci dare la forma dei tensori)
    with torch.no_grad():
        output_forma = modello.decoder(img_tensor).shape
    # output_forma è tipo (1, D, Altezza, Larghezza)
    capacita_totale = output_forma[1] * output_forma[2] * output_forma[3]

    # 3. Prepariamo il payload ridondante (IL TRUCCO)
    bit_messaggio = stringa_a_bit(testo_segreto)
    lunghezza_msg = len(bit_messaggio)

    ripetizioni = capacita_totale // lunghezza_msg
    bit_rimanenti = capacita_totale % lunghezza_msg

    print(f"Capacità dell'immagine : {capacita_totale} bit")
    print(f"Dimensione del tuo link: {lunghezza_msg} bit")
    print(f"Il link verrà ripetuto : {ripetizioni} volte dentro l'immagine!\n")

    # Riempiamo lo spazio moltiplicando la lista e aggiungendo il resto
    payload_completo = (bit_messaggio * ripetizioni) + bit_messaggio[:bit_rimanenti]

    # Creiamo il tensore matematico da dare in pasto all'Encoder
    data_tensor = torch.tensor(payload_completo, dtype=torch.float32).view(output_forma).to(modello.device)

    # 4. ENCODING RAW (Nascondiamo i dati bypassando la libreria)
    print("Iniezione diretta nei tensori...")
    with torch.no_grad():
        img_stego_tensor = modello.encoder(img_tensor, data_tensor)

    # Riconvertiamo per poter aggiungere il rumore
    img_stego_arr = (img_stego_tensor.squeeze().cpu().numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)

    # 5. IL TRAUMA (Aggiunta del rumore)
    print("Aggiunta del rumore...")
    rumore = np.random.normal(0, sigma_rumore, img_stego_arr.shape)
    img_corrotta_arr = np.clip(img_stego_arr + rumore, 0, 255).astype(np.uint8)

    # 6. DECODING RAW
    print("Estrazione cruda dei bit...")
    img_corrotta_tensor = transforms.ToTensor()(Image.fromarray(img_corrotta_arr)).unsqueeze(0).to(modello.device)

    with torch.no_grad():
        mappa_probabilita = modello.decoder(img_corrotta_tensor)

    bit_estratti_totali = (mappa_probabilita > 0).cpu().numpy().flatten().astype(int)

    # 7. LA MAGIA: IL VOTO A MAGGIORANZA
    print("Applicazione del processo democratico per correggere gli errori...\n")

    # Prendiamo solo i bit perfetti che compongono le nostre 'N' copie
    bit_utili = bit_estratti_totali[:ripetizioni * lunghezza_msg]

    # Rimodelliamo in una matrice dove ogni riga è una copia del messaggio
    matrice_voti = bit_utili.reshape((ripetizioni, lunghezza_msg))

    # Facciamo la media per ogni colonna. Se la media è > 0.5, la maggioranza ha votato "1"!
    bit_finali_votati = (np.mean(matrice_voti, axis=0) > 0.5).astype(int).tolist()

    testo_recuperato = bit_a_stringa(bit_finali_votati)

    print("--- RISULTATO FINALE ---")
    print(f"Messaggio Originale: {testo_segreto}")
    print(f"Messaggio Estratto : {testo_recuperato}")

    if testo_segreto == testo_recuperato:
        print("\n🟢 VITTORIA TOTALE! L'intelligenza collettiva ha riparato i danni del rumore.")
    else:
        print("\n🟡 QUASI! Il rumore ha battuto alcune votazioni, ma guarda quanto testo è intatto rispetto al crash di prima!")

# ESECUZIONE
# Assicurati di avere il percorso immagine e il modello pronti
messaggio_segreto = "https://it.wikipedia.org/wiki/Pongo_pygmaeus"
hacking_steganografico(steganogan_model, 'research/input_512.png', messaggio_segreto, sigma_rumore=5.0)

--- INIZIO HACKING STEGANOGRAFICO (Forza Bruta + Voto) ---
Livello di Rumore (Sigma Gaussiano): 5.0

Capacità dell'immagine : 2097152 bit
Dimensione del tuo link: 352 bit
Il link verrà ripetuto : 5957 volte dentro l'immagine!

Iniezione diretta nei tensori...
Aggiunta del rumore...
Estrazione cruda dei bit...
Applicazione del processo democratico per correggere gli errori...

--- RISULTATO FINALE ---
Messaggio Originale: https://it.wikipedia.org/wiki/Pongo_pygmaeus
Messaggio Estratto : https:/'it.wikipefia.org/wiki/Pongo_pygmauus

🟡 QUASI! Il rumore ha battuto alcune votazioni, ma guarda quanto testo è intatto rispetto al crash di prima!


In [21]:
import torch
import torchvision.transforms as transforms
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from PIL import Image

def stringa_a_bit(testo):
    """Converte una stringa in una lista di bit (0 e 1)"""
    return [int(b) for b in ''.join(format(ord(c), '08b') for c in testo)]

def calcola_metriche_forza_bruta(modello, percorso_input, testo_segreto):
    print("--- GENERAZIONE IMMAGINE (Forza Bruta / 100% Capacità) ---")

    # 1. Carichiamo l'immagine originale come array Numpy e Tensore PyTorch
    img_pil = Image.open(percorso_input).convert('RGB')
    img_orig_arr = np.array(img_pil)
    img_tensor = transforms.ToTensor()(img_pil).unsqueeze(0).to(modello.device)

    # 2. Prepariamo il payload ridondante
    with torch.no_grad():
        output_forma = modello.decoder(img_tensor).shape

    capacita_totale = output_forma[1] * output_forma[2] * output_forma[3]
    bit_messaggio = stringa_a_bit(testo_segreto)
    lunghezza_msg = len(bit_messaggio)

    ripetizioni = capacita_totale // lunghezza_msg
    bit_rimanenti = capacita_totale % lunghezza_msg

    print(f"Capacità dell'immagine : {capacita_totale} bit (Riempimento Totale)")

    payload_completo = (bit_messaggio * ripetizioni) + bit_messaggio[:bit_rimanenti]
    data_tensor = torch.tensor(payload_completo, dtype=torch.float32).view(output_forma).to(modello.device)

    # 3. ENCODING RAW (Generazione Immagine Stego)
    with torch.no_grad():
        img_stego_tensor = modello.encoder(img_tensor, data_tensor)

    # Convertiamo l'output del modello (Tensore) in un'immagine (Array Numpy)
    img_stego_arr = (img_stego_tensor.squeeze().cpu().numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)

    # Salviamo l'immagine per eventuali ispezioni visive
    percorso_output = 'output_forza_bruta.png'
    Image.fromarray(img_stego_arr).save(percorso_output)
    print(f"Immagine stego salvata in: {percorso_output}\n")

    # 4. CALCOLO DELLE METRICHE
    print("--- METRICHE DI QUALITÀ VISIVA (vs. Originale) ---")

    # MSE: Media degli errori quadratici calcolata pixel per pixel tra le due immagini [cite: 210]
    mse = np.mean((img_orig_arr.astype(np.float64) - img_stego_arr.astype(np.float64)) ** 2)

    # PSNR: Calcolato in decibel (dB) usando il massimo valore possibile per il pixel (255) [cite: 209, 212]
    valore_psnr = psnr(img_orig_arr, img_stego_arr, data_range=255)

    # SSIM: Indice di somiglianza strutturale (1.0 = identiche) [cite: 224, 225]
    valore_ssim = ssim(img_orig_arr, img_stego_arr, channel_axis=-1, data_range=255)

    print(f"MSE  : {mse:.4f}")
    print(f"PSNR : {valore_psnr:.4f} dB")
    print(f"SSIM : {valore_ssim:.4f}")

# ESECUZIONE
# Assicurati di aver definito 'steganogan_model' precedentemente nel notebook
messaggio_segreto = "https://it.wikipedia.org/wiki/Pongo_pygmaeus"
calcola_metriche_forza_bruta(steganogan_model, 'research/input_512.png', messaggio_segreto)

--- GENERAZIONE IMMAGINE (Forza Bruta / 100% Capacità) ---
Capacità dell'immagine : 2097152 bit (Riempimento Totale)
Immagine stego salvata in: output_forza_bruta.png

--- METRICHE DI QUALITÀ VISIVA (vs. Originale) ---
MSE  : 57.1876
PSNR : 30.5578 dB
SSIM : 0.8148
